# От ноутбука к версии модели

Цель: обучить простую модель оттока и сохранить **кандидата** в MLflow. Данные синтетические; Airflow и PostgreSQL для этого шага не нужны. Запускайте ячейки сверху вниз.

## 1. Подключиться к MLflow

Из контейнера JupyterLab сервер MLflow доступен по имени `mlflow`. В браузере его интерфейс открыт на [localhost:5000](http://localhost:5000).

In [ ]:
import os

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("notebook-churn-demo")
print("MLflow:", mlflow.get_tracking_uri())

## 2. Подготовить данные

У каждого клиента три признака. `churn` — исторический факт оттока, который используется только при обучении и проверке.

In [ ]:
rng = np.random.default_rng(117)
clients = pd.DataFrame({
    "tenure_months": rng.integers(1, 73, 1400),
    "monthly_spend": rng.uniform(10, 200, 1400).round(2),
    "support_tickets": rng.integers(0, 9, 1400),
})
logit = -1.8 - 0.035 * clients.tenure_months + 0.012 * clients.monthly_spend + 0.42 * clients.support_tickets
churn = (rng.random(len(clients)) < 1 / (1 + np.exp(-logit))).astype(int)
clients.assign(churn=churn).head()

## 3. Обучить и проверить

Часть клиентов оставляем для проверки качества. ROC AUC показывает, насколько модель разделяет клиентов с оттоком и без него.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    clients, churn, test_size=0.25, random_state=117, stratify=churn
)
model = LogisticRegression(max_iter=500)
model.fit(x_train, y_train)
roc_auc = roc_auc_score(y_test, model.predict_proba(x_test)[:, 1])
print(f"ROC AUC: {roc_auc:.3f}")
assert roc_auc >= 0.65, "Качество ниже порога для демонстрации"

## 4. Сохранить кандидата

MLflow запишет параметры, метрику и модель. Сам файл модели попадёт в S3 через сервер MLflow. Регистрация создаст следующую версию `churn-prediction`, но **не переключит** alias `production`.

In [ ]:
with mlflow.start_run(run_name="jupyter-logistic-regression") as run:
    mlflow.set_tags({
        "training_source": "notebook",
        "model_contract": "churn-v1",
        "dataset_version": "notebook-synthetic-v1",
    })
    mlflow.log_params({"algorithm": "LogisticRegression", "train_rows": len(x_train)})
    mlflow.log_metric("roc_auc", float(roc_auc))
    logged = mlflow.sklearn.log_model(
        model, name="model", input_example=x_test.head(2),
        signature=infer_signature(x_test, model.predict(x_test)),
    )
    version = mlflow.register_model(logged.model_uri, "churn-prediction")
    client = mlflow.MlflowClient()
    client.set_model_version_tag("churn-prediction", version.version, "validation_status", "passed")
    client.set_model_version_tag("churn-prediction", version.version, "model_contract", "churn-v1")

print(f"Run: {run.info.run_id}")
print(f"Новая версия churn-prediction: v{version.version}")
print("Alias production не менялся")

## 5. Проверить результат

Откройте [MLflow → Models → churn-prediction](http://localhost:5000/#/models/churn-prediction). Найдите новую версию и её run в эксперименте `notebook-churn-demo`. В [S3-консоли](http://localhost:9001) откройте bucket `mlflow-artifacts` и файлы модели.

Дальше можно сравнить кандидата с текущей версией и вручную назначить ему alias `production` в MLflow. После этого откройте [ML-сервис `/model`](http://localhost:8000/model) и сделайте прогноз через [API](http://localhost:8000/docs). Повторный запуск ячеек создаст ещё один run и новую версию — это ожидаемо.